This code generates the combined csv that are produced by the pruning and benchmarking scripts.

In [ ]:
import pandas as pd
import os

# Define the base directory where your Vision folder is located
base_dir = '/Users/arihangupta/Downloads/pruning_project_data/Vision'

# Define the datasets to process
datasets = ['bloodmnist', 'dermamnist', 'pathmnist']

# Create name mapping dictionary
# Maps benchmarking file names to pruning file names
name_mapping = {
    # Baseline models - position of "baseline" is different
    'vit_base_patch16_224_baseline': 'baseline_vit_base_patch16_224',
    'vit_small_patch16_224_baseline': 'baseline_vit_small_patch16_224',
    'vit_tiny_patch16_224_baseline': 'baseline_vit_tiny_patch16_224',
    
    # KD models - exact match (no mapping needed, but included for clarity)
    'kd_vit_base_patch16_224_to_vit_small_patch16_224': 'kd_vit_base_patch16_224_to_vit_small_patch16_224',
    'kd_vit_base_patch16_224_to_vit_tiny_patch16_224': 'kd_vit_base_patch16_224_to_vit_tiny_patch16_224',
    'kd_vit_small_patch16_224_to_vit_tiny_patch16_224': 'kd_vit_small_patch16_224_to_vit_tiny_patch16_224',
    
    # Quantization models - exact match (no mapping needed, but included for clarity)
    'quantization_vit_base_patch16_224': 'quantization_vit_base_patch16_224',
    'quantization_vit_small_patch16_224': 'quantization_vit_small_patch16_224',
    'quantization_vit_tiny_patch16_224': 'quantization_vit_tiny_patch16_224',
}

for dataset in datasets:
    print(f"\nProcessing {dataset}...")
    
    # Define file paths
    benchmarking_file = os.path.join(base_dir, f'benchmarking_results/{dataset}_results.csv')
    pruning_file = os.path.join(base_dir, f'pruned_models/{dataset}/{dataset}_pruning_results.csv')
    output_file = os.path.join(base_dir, f'{dataset}_combined_results.csv')
    
    # Check if files exist
    if not os.path.exists(benchmarking_file):
        print(f"  Warning: {benchmarking_file} not found, skipping...")
        continue
    if not os.path.exists(pruning_file):
        print(f"  Warning: {pruning_file} not found, skipping...")
        continue
    
    # Read the CSV files
    df_benchmarking = pd.read_csv(benchmarking_file)
    df_pruning = pd.read_csv(pruning_file)
    
    # Remove any unnamed columns
    df_benchmarking = df_benchmarking.loc[:, ~df_benchmarking.columns.str.contains('^Unnamed')]
    df_pruning = df_pruning.loc[:, ~df_pruning.columns.str.contains('^Unnamed')]
    
    print(f"  Benchmarking file shape: {df_benchmarking.shape}")
    print(f"  Pruning file shape: {df_pruning.shape}")
    
    # Create a mapped column in the benchmarking dataframe
    df_benchmarking['mapped_model_name'] = df_benchmarking['model_name'].map(name_mapping)
    
    # Check if any models didn't get mapped
    unmapped = df_benchmarking[df_benchmarking['mapped_model_name'].isna()]['model_name'].unique()
    if len(unmapped) > 0:
        print(f"  WARNING: Some models were not mapped: {unmapped}")
    
    # Perform inner join on mapped_model_name (from benchmarking) and Variant (from pruning)
    df_combined = pd.merge(
        df_benchmarking, 
        df_pruning, 
        left_on='mapped_model_name', 
        right_on='Variant', 
        how='inner'
    )
    
    # Drop the temporary mapped_model_name column
    df_combined = df_combined.drop(columns=['mapped_model_name'])
    
    print(f"  Combined file shape: {df_combined.shape}")
    print(f"  Matched {len(df_combined)} rows")
    
    # Save the combined file
    df_combined.to_csv(output_file, index=False)
    print(f"  Saved to: {output_file}")

print("\nAll datasets processed!")

This code works to average all of the results (now in one file per dataset) and also store the the SD. This merged result is then used for visualization.

In [ ]:
import pandas as pd
import os
import numpy as np
import re

# ----------------------------------------------------------------------
# CONFIGURATION
# ----------------------------------------------------------------------
base_dir = '/Users/arihangupta/Downloads/pruning_project_data/Vision'
output_dir = os.path.join(base_dir, 'merged_results')
os.makedirs(output_dir, exist_ok=True)

datasets = ['bloodmnist', 'dermamnist', 'pathmnist']

columns_to_average = [
    'Acc', 'AUC', 'Precision', 'Recall', 'Loss',
    'InferenceTime_per_batch_s', 'PeakRAM_MB',
    'RetrainEnergy_kWh', 'RetrainEmissions_kg',
    'images_processed', 'elapsed_s', 'throughput_imgs_per_s',
    'auc', 'median_batch_ms', 'p50_ms', 'p90_ms',
    'peak_gpu_mem_MB', 'avg_power_W',
    'energy_kWh_total', 'energy_kWh_per_batch', 'energy_kWh_per_image',
    'emissions_kg_total', 'cpu_power_w', 'gpu_power_w', 'ram_power_w'
]

grouping_columns = [
    'model_name', 'pruning_method', 'sparsity',
    'stored_precision', 'batch_size', 'runtime_precision'
]

# ----------------------------------------------------------------------
# PROCESS EACH DATASET
# ----------------------------------------------------------------------
for dataset in datasets:
    print(f"\nProcessing {dataset}...")

    input_file = os.path.join(base_dir, f'{dataset}_combined_results.csv')
    if not os.path.exists(input_file):
        print(f"  Warning: {input_file} not found, skipping...")
        continue

    df = pd.read_csv(input_file)
    original_count = len(df)
    print(f"  Original shape: {df.shape}")

    # --------------------------------------------------------------
    # 1. Verify model size column
    # --------------------------------------------------------------
    if 'ModelSizeMB' not in df.columns:
        print("  Error: 'ModelSizeMB' column missing!")
        continue

    # --------------------------------------------------------------
    # 2. DELETE: baseline pruning_method + mismatched precision
    # --------------------------------------------------------------
    delete_mask = (
        (df['pruning_method'] == 'baseline') &
        (df['stored_precision'] != df['runtime_precision'])
    )
    deleted_count = delete_mask.sum()
    df = df[~delete_mask].copy()
    print(f"  Deleted {deleted_count} baseline rows with mismatched precision")

    # --------------------------------------------------------------
    # 3. Build lookup: quantized model → size (MB)
    # --------------------------------------------------------------
    quant_names = [
        'quantization_vit_base_patch16_224',
        'quantization_vit_small_patch16_224',
        'quantization_vit_tiny_patch16_224'
    ]

    quant_model_sizes = {}
    for qname in quant_names:
        mask = df['model_name'].str.contains(qname, na=False)
        if mask.any():
            size = df.loc[mask, 'ModelSizeMB'].iloc[0]
            quant_model_sizes[qname] = size
            print(f"    Quantized size: {qname} = {size:.2f} MB")
        else:
            print(f"    Warning: No rows for quantized model '{qname}'")

    # --------------------------------------------------------------
    # 4. Add precision_status
    # --------------------------------------------------------------
    def classify_precision(row):
        if any(q in row['model_name'] for q in quant_names):
            return "quantized"
        return "baseline" if row['stored_precision'] == row['runtime_precision'] else "quantized"

    df['precision_status'] = df.apply(classify_precision, axis=1)

    # --------------------------------------------------------------
    # 5. FIX ModelSizeMB for ALL quantized rows
    # --------------------------------------------------------------
    def fix_model_size(row):
        if row['precision_status'] == 'baseline':
            return row['ModelSizeMB']

        model_name = row['model_name']

        # 1. KD-style: kd_*_to_vit_*_patch16_224
        kd_match = re.search(r'kd_.*_to_(vit_(?:base|small|tiny)_patch16_224)', model_name)
        if kd_match:
            target = kd_match.group(1)
            quant_key = f"quantization_{target}"
            if quant_key in quant_model_sizes:
                new_sz = quant_model_sizes[quant_key]
                print(f"    KD fix: {model_name} → {quant_key} ({new_sz:.2f} MB)")
                return new_sz

        # 2. Any other quantized run (e.g. baseline model with amp/int8)
        base_match = re.search(r'(vit_(?:base|small|small|tiny)_patch16_224)', model_name)
        if base_match:
            base_part = base_match.group(1)
            quant_key = f"quantization_{base_part}"
            if quant_key in quant_model_sizes:
                new_sz = quant_model_sizes[quant_key]
                print(f"    Quant fix: {model_name} → {quant_key} ({new_sz:.2f} MB)")
                return new_sz

        return row['ModelSizeMB']

    df['ModelSizeMB'] = df.apply(fix_model_size, axis=1)

    # --------------------------------------------------------------
    # 6. Group-by and aggregate
    # --------------------------------------------------------------
    full_group_cols = grouping_columns + ['precision_status']

    agg_dict = {}
    for col in columns_to_average:
        if col in df.columns:
            agg_dict[col] = ['mean', 'std']

    agg_dict['ModelSizeMB'] = ['mean', 'std']

    for col in df.columns:
        if col not in full_group_cols and col not in columns_to_average and col not in ['run_id', 'rep', 'ModelSizeMB']:
            agg_dict[col] = 'first'

    df_averaged = df.groupby(full_group_cols, as_index=False).agg(agg_dict)

    # --------------------------------------------------------------
    # 7. Flatten multi-level columns
    # --------------------------------------------------------------
    new_columns = []
    for col in df_averaged.columns:
        if isinstance(col, tuple):
            if col[1] == 'mean':
                new_columns.append(col[0])
            elif col[1] == 'std':
                new_columns.append(f"{col[0]}_sd")
            else:
                new_columns.append(col[0])
        else:
            new_columns.append(col)

    df_averaged.columns = new_columns

    # --------------------------------------------------------------
    # 8. Re-order: metric then _sd
    # --------------------------------------------------------------
    final_columns = []
    processed = set()
    for col in df_averaged.columns:
        if col not in processed:
            final_columns.append(col)
            processed.add(col)
            sd_col = f"{col}_sd"
            if sd_col in df_averaged.columns and sd_col not in processed:
                final_columns.append(sd_col)
                processed.add(sd_col)

    df_averaged = df_averaged[final_columns]

    print(f"  Final shape after processing: {df_averaged.shape}")
    print(f"  Total rows: {original_count} → {len(df)} kept → {len(df_averaged)} groups")

    # --------------------------------------------------------------
    # 9. Save
    # --------------------------------------------------------------
    output_file = os.path.join(output_dir, f'{dataset}_averaged_results.csv')
    df_averaged.to_csv(output_file, index=False)
    print(f"  Saved to: {output_file}")

print("\nAll datasets processed!")
print(f"Results saved to: {output_dir}")

Vision Transformer Radar chart creation along with LaTeX Table. 

In [ ]:
"""
Vision Transformer Radar Chart Visualization with LaTeX Table Generation

This script creates radar charts for Vision Transformer models (Base, Small, Tiny)
and generates comprehensive LaTeX tables with raw and normalized metrics.

Usage:
    python vision_radar_with_latex.py
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from math import pi
import os


def create_method_label(row):
    """Create display labels for each method variant."""
    method = row['pruning_method']
    precision = row['runtime_precision']
    variant = row['Variant']
    
    # BASE chart: Shows all optimizations derived from baseline_vit_base_patch16_224
    # SMALL chart: Shows all optimizations derived from baseline_vit_small_patch16_224
    # TINY chart: Shows all optimizations derived from baseline_vit_tiny_patch16_224
    
    if variant == 'baseline_vit_base_patch16_224' and precision == 'fp32':
        return 'Baseline (Base)', 'baseline', 0, 'Base'
    
    elif variant == 'baseline_vit_small_patch16_224' and method == 'baseline' and precision == 'fp32':
        return 'Baseline (Small)', 'baseline', 0, 'Small'
    
    elif variant == 'baseline_vit_tiny_patch16_224' and method == 'baseline' and precision == 'fp32':
        return 'Baseline (Tiny)', 'baseline', 0, 'Tiny'
    
    # Quantized versions - grouped by which baseline was quantized
    elif method == 'quantization' and precision == 'amp':
        if 'vit_base_patch' in variant:
            return 'Quantized (Base)', 'quantized', 1, 'Base'
        elif 'vit_small_patch' in variant:
            return 'Quantized (Small)', 'quantized', 1, 'Small'
        elif 'vit_tiny_patch' in variant:
            return 'Quantized (Tiny)', 'quantized', 1, 'Tiny'
    
    # Knowledge Distillation - grouped by TEACHER (source) model
    elif method == 'kd' and precision == 'fp32':
        if 'base' in variant and 'small' in variant:
            return 'KD (Base→Small)', 'kd', 2, 'Base'
        elif 'base' in variant and 'tiny' in variant:
            return 'KD (Base→Tiny)', 'kd', 3, 'Base'
        elif 'small' in variant and 'tiny' in variant:
            return 'KD (Small→Tiny)', 'kd', 2, 'Small'
        else:
            return 'KD', 'kd', 99, 'Unknown'
    
    # KD with quantization (hybrid) - grouped by TEACHER model
    elif method == 'kd' and precision == 'amp':
        if 'base' in variant and 'small' in variant:
            return 'KD+Quant (Base→Small)', 'kd_quant', 4, 'Base'
        elif 'base' in variant and 'tiny' in variant:
            return 'KD+Quant (Base→Tiny)', 'kd_quant', 5, 'Base'
        elif 'small' in variant and 'tiny' in variant:
            return 'KD+Quant (Small→Tiny)', 'kd_quant', 3, 'Small'
        else:
            return 'KD+Quant', 'kd_quant', 99, 'Unknown'
    
    else:
        # Try to determine group from variant name
        if 'vit_base_patch' in variant:
            return f'{method} {precision}', 'other', 99, 'Base'
        elif 'vit_small_patch' in variant:
            return f'{method} {precision}', 'other', 99, 'Small'
        elif 'vit_tiny_patch' in variant:
            return f'{method} {precision}', 'other', 99, 'Tiny'
        else:
            return f'{method} {precision}', 'other', 99, 'Unknown'


def get_method_color(label):
    """Get color for each method label"""
    color_map = {
        # Baselines - red shades
        'Baseline (Base)': '#FF0000',
        'Baseline (Small)': '#FF0000',
        'Baseline (Tiny)': '#FF0000',
        
        # Quantized models - green shades  
        'Quantized (Base)': '#228B22',
        'Quantized (Small)': '#228B22',
        'Quantized (Tiny)': '#228B22',
        
        # KD methods - blue shades
        'KD (Base→Small)': '#0000CD',
        'KD (Base→Tiny)': '#4169E1',
        'KD (Small→Tiny)': '#0000CD',
        
        # KD+Quantization (hybrid) - purple shades
        'KD+Quant (Base→Small)': '#6A0DAD',
        'KD+Quant (Base→Tiny)': '#9370DB',
        'KD+Quant (Small→Tiny)': '#6A0DAD',
    }
    return color_map.get(label, '#333333')


def save_latex_table(models_to_plot, baseline, max_ratios, output_path, model_size, batch_size, dataset_name):
    """Generate LaTeX tables with raw and normalized metrics."""
    
    with open(output_path, 'w') as f:
        # Write header
        f.write("% Raw data for Vision Transformer radar chart\n")
        f.write(f"% Dataset: {dataset_name}, Model Size: {model_size}, Batch Size: {batch_size}\n\n")
        
        # Write baseline values
        f.write("% Baseline Values\n")
        f.write("\\begin{table}[h]\n")
        f.write("\\centering\n")
        f.write("\\begin{tabular}{ll}\n")
        f.write("\\toprule\n")
        f.write("Metric & Value \\\\\n")
        f.write("\\midrule\n")
        f.write(f"Accuracy & {baseline['Acc']:.4f} \\\\\n")
        f.write(f"AUC & {baseline['AUC']:.6f} \\\\\n")
        f.write(f"Throughput (img/s) & {baseline['throughput_imgs_per_s']:.2f} \\\\\n")
        f.write(f"Energy (kWh/img) & {baseline['energy_kWh_per_image']:.2e} \\\\\n")
        f.write(f"Peak RAM (MB) & {baseline['PeakRAM_MB']:.1f} \\\\\n")
        f.write(f"Model Size (MB) & {baseline['ModelSizeMB']:.1f} \\\\\n")
        f.write("\\bottomrule\n")
        f.write("\\end{tabular}\n")
        f.write(f"\\caption{{Baseline Values - {model_size} Model}}\n")
        f.write("\\end{table}\n\n")
        
        # Write raw metrics table
        f.write("% Raw Metrics for All Models\n")
        f.write("\\begin{table}[h]\n")
        f.write("\\centering\n")
        f.write("\\small\n")
        f.write("\\begin{tabular}{lcccccc}\n")
        f.write("\\toprule\n")
        f.write("Method & Acc & AUC & Throughput & Energy & RAM & Size \\\\\n")
        f.write(" & & & (img/s) & (kWh/img) & (MB) & (MB) \\\\\n")
        f.write("\\midrule\n")
        
        for model in models_to_plot:
            raw = model['raw_values']
            method_name = model['label'].replace('_', '\\_').replace('%', '\\%').replace('&', '\\&').replace('→', '$\\rightarrow$')
            f.write(f"{method_name} & {raw['Acc']:.4f} & {raw['AUC']:.6f} & "
                   f"{raw['Throughput']:.2f} & {raw['Energy']:.2e} & "
                   f"{raw['RAM']:.1f} & {raw['ModelSize']:.1f} \\\\\n")
        
        f.write("\\bottomrule\n")
        f.write("\\end{tabular}\n")
        f.write(f"\\caption{{Raw Performance Metrics - {model_size} Model}}\n")
        f.write("\\end{table}\n\n")
        
        # Write normalized values table
        f.write("% Normalized Values for Radar Plot (0-1 scale)\n")
        f.write("\\begin{table}[h]\n")
        f.write("\\centering\n")
        f.write("\\small\n")
        f.write("\\begin{tabular}{lcccccc}\n")
        f.write("\\toprule\n")
        f.write("Method & Acc & AUC & Throughput & Energy & RAM & Size \\\\\n")
        f.write(" & (norm) & (norm) & (ratio) & (ratio) & (ratio) & (ratio) \\\\\n")
        f.write("\\midrule\n")
        
        for model in models_to_plot:
            method_name = model['label'].replace('_', '\\_').replace('%', '\\%').replace('&', '\\&').replace('→', '$\\rightarrow$')
            vals = model['values']
            f.write(f"{method_name} & {vals[0]:.4f} & {vals[1]:.4f} & "
                   f"{vals[2]:.4f} & {vals[3]:.4f} & {vals[4]:.4f} & {vals[5]:.4f} \\\\\n")
        
        f.write("\\bottomrule\n")
        f.write("\\end{tabular}\n")
        f.write(f"\\caption{{Normalized Values for Radar Chart (Baseline = 1.0) - {model_size} Model}}\n")
        f.write("\\end{table}\n\n")
        
        # Write scaling factors
        f.write("% Scaling Factors Used\n")
        f.write("\\begin{table}[h]\n")
        f.write("\\centering\n")
        f.write("\\begin{tabular}{ll}\n")
        f.write("\\toprule\n")
        f.write("Metric & Max Ratio \\\\\n")
        f.write("\\midrule\n")
        f.write(f"Accuracy & {max_ratios['acc']:.4f} (absolute) \\\\\n")
        f.write(f"AUC & {max_ratios['auc']:.6f} (absolute) \\\\\n")
        f.write(f"Throughput & {max_ratios['throughput']:.2f}x baseline \\\\\n")
        f.write(f"Energy & {max_ratios['energy']:.2f}x (baseline/current) \\\\\n")
        f.write(f"RAM & {max_ratios['ram']:.2f}x (baseline/current) \\\\\n")
        f.write(f"Model Size & {max_ratios['modelsize']:.2f}x (baseline/current) \\\\\n")
        f.write("\\bottomrule\n")
        f.write("\\end{tabular}\n")
        f.write(f"\\caption{{Scaling Factors - {model_size} Model}}\n")
        f.write("\\end{table}\n\n")
        
        # Write Python-readable format
        f.write("% Python-readable format for plotting\n")
        f.write("% models = [\n")
        for model in models_to_plot:
            f.write(f"%   {{\n")
            f.write(f"%     'label': '{model['label']}',\n")
            f.write(f"%     'normalized': {model['values']},\n")
            raw = model['raw_values']
            f.write(f"%     'raw': {{'Acc': {raw['Acc']:.4f}, 'AUC': {raw['AUC']:.6f}, ")
            f.write(f"'Throughput': {raw['Throughput']:.2f}, 'Energy': {raw['Energy']:.2e}, ")
            f.write(f"'RAM': {raw['RAM']:.1f}, 'ModelSize': {raw['ModelSize']:.1f}}}\n")
            f.write(f"%   }},\n")
        f.write("% ]\n")


def create_vision_radar_with_latex(csv_path, output_dir, dataset_name):
    """
    Create radar charts and LaTeX tables for Vision Transformer models.
    
    Processes Base, Small, and Tiny models separately, showing each baseline
    and all optimization methods derived from it.
    """
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Read the CSV file
    df = pd.read_csv(csv_path)
    
    print(f"\n{'='*80}")
    print(f"Processing Dataset: {dataset_name.upper()}")
    print(f"{'='*80}")
    print(f"Loaded CSV with {len(df)} rows")
    print(f"Columns: {df.columns.tolist()[:10]}...")
    
    # Check if batch_size column exists
    if 'batch_size' not in df.columns:
        print(f"\nAvailable columns: {df.columns.tolist()}")
        raise KeyError("'batch_size' column not found in CSV")
    
    # Filter for batch size 8
    df = df[df['batch_size'] == 8].copy()
    print(f"Filtered to {len(df)} rows with batch_size=8")
    
    # Add method labels and model size
    df[['method_label', 'method_group', 'sort_order', 'model_size']] = df.apply(
        lambda row: pd.Series(create_method_label(row)), axis=1
    )
    
    # Safety check: Convert percentages to decimals if needed
    if df['Acc'].max() > 1.5:
        print("Converting Accuracy from percentage to decimal...")
        df['Acc'] = df['Acc'] / 100.0
    if df['AUC'].max() > 1.5:
        print("Converting AUC from percentage to decimal...")
        df['AUC'] = df['AUC'] / 100.0
    
    # Safety check: Handle zero or NaN values
    df['energy_kWh_per_image'] = df['energy_kWh_per_image'].replace(0, np.nan)
    df['PeakRAM_MB'] = df['PeakRAM_MB'].replace(0, np.nan)
    df['ModelSizeMB'] = df['ModelSizeMB'].replace(0, np.nan)
    
    # Define metrics
    metrics = ['Acc', 'AUC', 'throughput_imgs_per_s', 'energy_kWh_per_image', 'PeakRAM_MB', 'ModelSizeMB']
    metric_labels = ['Accuracy', 'AUC', 'Throughput\n(imgs/s)', 'Energy\n(lower is better)', 
                    'Peak RAM\n(lower is better)', 'Model Size\n(lower is better)']
    
    # Process each model size separately
    model_sizes = ['Base', 'Small', 'Tiny']
    
    for model_size in model_sizes:
        print(f"\n{'='*60}")
        print(f"Creating radar chart and LaTeX tables for {model_size} baseline")
        print(f"{'='*60}")
        
        # Filter data for this model size
        size_df = df[df['model_size'] == model_size].copy()
        
        if size_df.empty:
            print(f"No data for {model_size} models, skipping...")
            continue
        
        # Get the baseline for this size
        if model_size == 'Base':
            baseline_candidates = size_df[size_df['method_label'] == 'Baseline (Base)']
        elif model_size == 'Small':
            baseline_candidates = size_df[size_df['method_label'] == 'Baseline (Small)']
        else:  # Tiny
            baseline_candidates = size_df[size_df['method_label'] == 'Baseline (Tiny)']
        
        if baseline_candidates.empty:
            print(f"No baseline found for {model_size} models, skipping...")
            continue
        
        # If multiple baselines, take the one with highest accuracy
        baseline = baseline_candidates.sort_values('Acc', ascending=False).iloc[0]
        
        print(f"Baseline values for {model_size}:")
        print(f"  Accuracy: {baseline['Acc']:.4f}")
        print(f"  AUC: {baseline['AUC']:.6f}")
        print(f"  Throughput: {baseline['throughput_imgs_per_s']:.2f} imgs/s")
        print(f"  Energy: {baseline['energy_kWh_per_image']:.2e} kWh/image")
        print(f"  RAM: {baseline['PeakRAM_MB']:.2f} MB")
        print(f"  Model Size: {baseline['ModelSizeMB']:.2f} MB")
        
        # Calculate ratios relative to baseline
        size_df['throughput_ratio'] = size_df['throughput_imgs_per_s'] / baseline['throughput_imgs_per_s']
        size_df['energy_ratio'] = baseline['energy_kWh_per_image'] / size_df['energy_kWh_per_image']
        size_df['ram_ratio'] = baseline['PeakRAM_MB'] / size_df['PeakRAM_MB']
        size_df['modelsize_ratio'] = baseline['ModelSizeMB'] / size_df['ModelSizeMB']
        
        # Get max values for scaling
        max_acc = 1.0
        max_auc = 1.0
        max_throughput_ratio = size_df['throughput_ratio'].max()
        max_energy_ratio = size_df['energy_ratio'].max()
        max_ram_ratio = size_df['ram_ratio'].max()
        max_modelsize_ratio = size_df['modelsize_ratio'].max()
        
        max_ratios = {
            'acc': max_acc,
            'auc': max_auc,
            'throughput': max_throughput_ratio,
            'energy': max_energy_ratio,
            'ram': max_ram_ratio,
            'modelsize': max_modelsize_ratio
        }
        
        print(f"\nMax ratios for scaling ({model_size}):")
        print(f"  Accuracy: {max_acc:.4f} (defined max)")
        print(f"  AUC: {max_auc:.6f} (defined max)")
        print(f"  Throughput ratio: {max_throughput_ratio:.2f}x baseline")
        print(f"  Energy ratio: {max_energy_ratio:.2f}x (baseline/current)")
        print(f"  RAM ratio: {max_ram_ratio:.2f}x (baseline/current)")
        print(f"  ModelSize ratio: {max_modelsize_ratio:.2f}x (baseline/current)")
        
        # Prepare data for plotting
        models_to_plot = []
        
        for _, row in size_df.iterrows():
            label = row['method_label']
            
            acc = row['Acc']
            auc = row['AUC']
            throughput = row['throughput_imgs_per_s']
            energy = row['energy_kWh_per_image']
            ram = row['PeakRAM_MB']
            modelsize = row['ModelSizeMB']
            
            # Skip if critical values are missing
            if pd.isna(acc) or pd.isna(auc) or pd.isna(throughput) or pd.isna(energy) or pd.isna(ram) or pd.isna(modelsize):
                print(f"  Skipping {label} due to missing values")
                continue
            
            # Normalize each metric
            normalized_values = []
            
            normalized_values.append(acc / max_acc)
            normalized_values.append(auc / max_auc)
            
            throughput_ratio = throughput / baseline['throughput_imgs_per_s']
            normalized_values.append(min(throughput_ratio / max_throughput_ratio, 1.0))
            
            energy_ratio = baseline['energy_kWh_per_image'] / energy
            normalized_values.append(min(energy_ratio / max_energy_ratio, 1.0))
            
            ram_ratio = baseline['PeakRAM_MB'] / ram
            normalized_values.append(min(ram_ratio / max_ram_ratio, 1.0))
            
            modelsize_ratio = baseline['ModelSizeMB'] / modelsize
            normalized_values.append(min(modelsize_ratio / max_modelsize_ratio, 1.0))
            
            models_to_plot.append({
                'label': label,
                'values': normalized_values,
                'raw_values': {
                    'Acc': acc,
                    'AUC': auc,
                    'Throughput': throughput,
                    'Energy': energy,
                    'RAM': ram,
                    'ModelSize': modelsize
                },
                'sort_order': row['sort_order']
            })
        
        # Sort by sort_order
        models_to_plot.sort(key=lambda x: x['sort_order'])
        
        # Save LaTeX table
        latex_filename = os.path.join(output_dir, f'{dataset_name}_{model_size.lower()}_batch8_data.tex')
        save_latex_table(models_to_plot, baseline, max_ratios, latex_filename, 
                        model_size, 8, dataset_name)
        print(f"\nLaTeX data file saved to: {latex_filename}")
        
        # Create the radar chart
        num_vars = len(metrics)
        angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
        angles += angles[:1]
        
        fig, ax = plt.subplots(figsize=(12, 10), subplot_kw=dict(projection='polar'))
        
        # Plot each model
        for model_data in models_to_plot:
            values = model_data['values']
            values += values[:1]
            
            linewidth = 3 if 'Baseline' in model_data['label'] else 2
            alpha_line = 1.0 if 'Baseline' in model_data['label'] else 0.8
            alpha_fill = 0.25 if 'Baseline' in model_data['label'] else 0.15
            
            color = get_method_color(model_data['label'])
            
            ax.plot(angles, values, 'o-', linewidth=linewidth, label=model_data['label'], 
                    color=color, markersize=8, alpha=alpha_line)
            ax.fill(angles, values, alpha=alpha_fill, color=color)
        
        # Set axis labels
        ax.set_xticks(angles[:-1])
        
        axis_labels = []
        metric_keys = ['Acc', 'AUC', 'throughput_imgs_per_s', 'energy_kWh_per_image', 'PeakRAM_MB', 'ModelSizeMB']
        for metric_label, metric_key in zip(metric_labels, metric_keys):
            baseline_val = baseline[metric_key]
            if metric_key == 'Acc':
                axis_labels.append(f"{metric_label}\n(baseline: {baseline_val:.3f})\n[absolute scale]")
            elif metric_key == 'AUC':
                axis_labels.append(f"{metric_label}\n(baseline: {baseline_val:.4f})\n[absolute scale]")
            elif metric_key == 'throughput_imgs_per_s':
                axis_labels.append(f"Throughput\n(baseline: {baseline_val:.1f} img/s)\n[ratio to baseline]")
            elif metric_key == 'energy_kWh_per_image':
                axis_labels.append(f"Energy\n(baseline: {baseline_val:.2e} kWh)\n[baseline/current]")
            elif metric_key == 'PeakRAM_MB':
                axis_labels.append(f"RAM\n(baseline: {baseline_val:.1f} MB)\n[baseline/current]")
            else:  # ModelSizeMB
                axis_labels.append(f"Model Size\n(baseline: {baseline_val:.1f} MB)\n[baseline/current]")
        
        ax.set_xticklabels(axis_labels, size=9)
        
        # Set y-axis
        ax.set_ylim(0, 1.0)
        ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
        ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], size=9)
        ax.grid(True, linestyle='--', alpha=0.7)
        
        # Add title
        dataset_display = dataset_name.replace('mnist', ' MNIST').title()
        plt.title(f'Vision Transformer {model_size} - {dataset_display} (Batch Size 8)\nAll Methods Normalized to {model_size} Baseline = 1.0',
                  size=14, weight='bold', pad=20)
        
        # Add legend
        plt.legend(loc='upper left', bbox_to_anchor=(1.05, 1.0), fontsize=10)
        
        # Save figure
        plot_filename = os.path.join(output_dir, f'vit_radar_{dataset_name}_{model_size.lower()}_batch8.png')
        plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
        print(f"Radar plot saved to: {plot_filename}")
        
        plt.show()
        plt.close()
        
        # Print summary table
        print(f"\n{model_size} Model Performance Summary (Batch Size 8)")
        print("="*120)
        print(f"{'Model':<30} {'Accuracy':<12} {'AUC':<12} {'Throughput':<15} {'Energy/img':<15} {'RAM (MB)':<12} {'Size (MB)':<12}")
        print("-"*120)
        
        for model in models_to_plot:
            raw = model['raw_values']
            print(f"{model['label']:<30} {raw['Acc']:<12.4f} {raw['AUC']:<12.6f} "
                  f"{raw['Throughput']:<15.2f} {raw['Energy']:<15.2e} {raw['RAM']:<12.1f} {raw['ModelSize']:<12.1f}")
        print("="*120)


if __name__ == "__main__":
    base_dir = "/Users/arihangupta/Downloads/pruning_project_data/Vision"
    merged_results_dir = os.path.join(base_dir, "merged_results")
    output_dir = os.path.join(base_dir, "Visuals")
    
    datasets = {
        'bloodmnist': os.path.join(merged_results_dir, "bloodmnist_averaged_results.csv"),
        'dermamnist': os.path.join(merged_results_dir, "dermamnist_averaged_results.csv"),
        'pathmnist': os.path.join(merged_results_dir, "pathmnist_averaged_results.csv")
    }
    
    for dataset_name, csv_path in datasets.items():
        print(f"\n{'#'*80}")
        print(f"# Processing {dataset_name.upper()}")
        print(f"{'#'*80}")
        
        try:
            create_vision_radar_with_latex(csv_path, output_dir, dataset_name)
        except Exception as e:
            print(f"Error processing {dataset_name}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    print(f"\n{'#'*80}")
    print("# Processing Complete!")
    print(f"# Output directory: {output_dir}")
    print(f"{'#'*80}")